# Is gradient clipping binding on VGG? --- measured, not inferred

`GRAD_CLIP_NORM = 10.0` runs identically for every model and phase -- one constant, one call site inside `_optimizer_step`, no model-type branching (`project/training_utils.py:519,544`). That much is certain from the code. What was never measured is whether the threshold **actually fires**: `clip_grad_norm_` returns the pre-clip total norm and the return value was discarded, so no record or log has ever carried it.

The one clue already in the code points toward VGG: clipping was added because *"VGG-19's depth forced a reduced learning rate to stay finite"* -- meaning unclipped VGG-19 gradients did exceed 10.0 and diverge. That is evidence it binds *sometimes*. It does not say whether it binds on 5% of steps (a safety net, changes nothing else) or 90% of steps (reshaping the whole descent, and a real competing explanation to the "no damage to repair" story in the paper for VGG-11's negative delta).

## What this measures

One BaCP magnitude cell each for `vgg11` and `resnet34` at 0.95 -- the point used throughout this project for "everything else identical" comparisons -- with gradient-norm telemetry turned on via `BACP_LOG_GRAD_NORM=1`. `nb_common`, `bacp.py` and `training_utils.py` gained a small opt-in log (off by default, so the sweep and the seed reruns are unaffected either way): every optimizer step appends its pre-clip norm, and `BaCPTrainer` prints one summary line per phase (contrastive, fine-tune) with the mean, p50/p90/p99, max, and the percentage of steps actually clamped.

Records land under a `gradnorm` variant suffix, separate from every other cell in the sweep, the seed reruns, and the LR probes.

In [ ]:
import sys, pathlib, os
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')

# Must be set BEFORE nb.run() spawns the training subprocess -- it reads this
# once at import time in training_utils.py.
os.environ['BACP_LOG_GRAD_NORM'] = '1'

import nb_common as nb
info = nb.setup()

## Sanity check

In [ ]:
SPARSITY = 0.95
SEED, GPU = 1, 0

cells = [
    nb.make_cell('vgg11',    'bacp', seed=SEED, pruner='magnitude',
                 sparsity=SPARSITY, variant='gradnorm'),
    nb.make_cell('resnet34', 'bacp', seed=SEED, pruner='magnitude',
                 sparsity=SPARSITY, variant='gradnorm'),
]
for c in cells:
    ref = nb.FAMILIES[c['config']['model_name']]['bacp']
    for k in ('epochs', 'epochs_ft', 'delta_T', 'val_split', 'tau',
              'learning_rate', 'contrastive_mode', 'proj_mode'):
        assert c['config'][k] == ref[k], f'{c["key"]}: {k} differs from FAMILIES'
    assert c['key'].endswith('.gradnorm'), f'{c["key"]} lacks the variant suffix'
    print(c['key'])

print(f'\n{len(cells)} cells; identical to the corresponding sweep cell in '
      f'every config field, differing only in the grad-norm telemetry.')
assert nb.sanity_check(cells), 'sanity check failed'

## Run --- sequentially

`run_parallel` was measured slower on this workload (0.87x aggregate throughput); see its docstring. ~35 min for vgg11, ~15 min for resnet34 at these settings, sequential.

In [ ]:
nb.run_group(cells, gpu=GPU)

## Reading the result

Grep the two `.log` files for `[grad-norm]` -- four lines total, one per (model, phase). The `clipped=` percentage is the answer: near 0% means clipping is a safety net that changes nothing on the steps that matter, and the negative VGG-11 delta stands as an architectural finding, not a clipping artefact. A high percentage on VGG and a low one on ResNet-34 would mean clipping reshapes most of VGG's updates, which is a genuine alternative explanation and would need to be said in the paper alongside, or instead of, the current one.

In [ ]:
import glob, os
root = os.environ['BACP_RESULTS_DIR']
for path in sorted(glob.glob(os.path.join(root, 'logs', '*.gradnorm.log'))):
    print(f'--- {os.path.basename(path)} ---')
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            if '[grad-norm]' in line:
                print(' ', line.strip())
    print()